# **Phase 4 – Gold Layer (Business Analytics)**

**Read the Silver Table**

In [0]:
silver_df = spark.table("workspace.default.silver_service_track")  

**Create a Temporary SQL View**

In [0]:
silver_df.createOrReplaceTempView("service_data")

In [0]:
%sql

SELECT *
FROM service_data
LIMIT 10;

device_id,customer_id,job_id,issue_type,job_status,received_date,promised_date,completed_date,technician_id,technician_name,repair_notes,estimated_cost,actual_cost,repair_duration,delivery_status,customer_name,phone_number,email,city,registration_date,brand,device_type,model_series,warranty_months,price_range
DEV019,CUST0295,JOB01181,Battery Issue,Completed,2024-01-29,2024-02-03,2024-01-31,T005,Kavitha Nair,Customer informed,4775.15,6845.21,2,On Time,Meera Khan,9995866934,meera.khan51@outlook.com,Pune,2023-04-09,Lenovo,Printer,Lenovo PRI-Series,12,Premium (> 40K)
DEV035,CUST0108,JOB00070,Keyboard Fault,Completed,2024-02-29,2024-03-05,2024-03-07,T006,Not Assigned,null,1450.68,4850.57,7,Delayed,Rahul Bajaj,9423573322,rahul.bajaj10@outlook.com,Bhopal,2022-04-14,Motorola,Laptop,Motorola LAP-Series,24,Mid-range (15K-40K)
DEV020,CUST0001,JOB00991,Software Crash,Cancelled,2024-03-13,2024-03-18,2024-03-17,T007,Meena Reddy,Spare part ordered,617.6,null,4,On Time,Kiran Patel,9433218196,kiran.patel5@gmail.com,Delhi,2022-08-12,Lenovo,Smartphone,Lenovo SMA-Series,12,Budget (< 15K)
DEV022,CUST0273,JOB01238,Hard Disk Failure,Pending,2024-01-23,2024-01-28,null,T008,Arjun Iyer,Returned to customer,3234.8,null,null,On Time,Shivam Singh,9074329271,shivam.singh17@hotmail.com,Lucknow,2023-03-16,Asus,Refrigerator,Asus REF-Series,18,Budget (< 15K)
DEV039,CUST0152,JOB00956,Overheating,Completed,2024-01-28,2024-02-02,2024-01-31,T005,Kavitha Nair,Returned to customer,5936.46,4845.37,3,On Time,Nisha Gupta,9874315274,nisha.gupta92@yahoo.com,Mumbai,2022-12-12,Realme,Laptop,Realme LAP-Series,6,Premium (> 40K)
DEV033,CUST0276,JOB00508,Charging Port Fault,In Progress,2024-02-05,2024-02-10,null,T004,Not Assigned,Customer follow-up pending,7955.21,null,null,On Time,Vinay Dubey,9789856257,vinay.dubey46@yahoo.com,Bengaluru,2023-02-18,LG,Smartphone,LG SMA-Series,6,Budget (< 15K)
DEV019,CUST0034,JOB00125,Software Crash,Completed,2024-01-16,2024-01-21,2024-01-19,T003,Priya Singh,null,1836.79,2000.1,3,On Time,Imran Verma,9872148951,imran.verma97@yahoo.com,Lucknow,2022-11-14,Lenovo,Printer,Lenovo PRI-Series,12,Premium (> 40K)
DEV031,CUST0142,JOB01130,Overheating,Completed,2024-01-21,2024-01-26,2024-01-23,T005,Kavitha Nair,Out of warranty,5232.72,3553.55,2,On Time,Yash Pandey,9705516940,yash.pandey77@hotmail.com,Pune,2023-11-03,LG,Laptop,LG LAP-Series,18,Budget (< 15K)
DEV021,CUST0232,JOB00303,Hard Disk Failure,In Progress,2024-01-28,2024-02-02,null,T003,Priya Singh,Customer follow-up pending,2720.73,null,null,On Time,Anjali Iyer,9490352135,anjali.iyer51@gmail.com,Lucknow,2022-11-03,Lenovo,Monitor,Lenovo MON-Series,18,Mid-range (15K-40K)
DEV043,CUST0181,JOB01475,Motherboard Failure,Completed,2024-01-31,2024-02-05,2024-02-05,T004,Amit Patel,Under warranty,5191.51,6972.09,5,On Time,Deepika Malhotra,9999650477,deepika.malhotra30@outlook.com,Kolkata,2023-01-20,Oppo,Microwave,Oppo MIC-Series,6,Mid-range (15K-40K)


**Technician Performance**
Q1. Which technician completed the highest number of service jobs?

In [0]:
from pyspark.sql.functions import count

technician_performance = silver_df.groupBy("technician_name") \
    .agg(
        count("job_id").alias("total_jobs")
    ) \
    .orderBy("total_jobs", ascending=False)

display(technician_performance)

technician_name,total_jobs
Priya Singh,203
Meena Reddy,199
Kavitha Nair,182
Suresh Rao,182
Deepak Sharma,178
Rajesh Kumar,175
Amit Patel,154
Arjun Iyer,152
Not Assigned,75


In [0]:
%sql

SELECT
technician_name,
COUNT(job_id) AS total_jobs

FROM service_data

GROUP BY technician_name

ORDER BY total_jobs DESC;

technician_name,total_jobs
Priya Singh,203
Meena Reddy,199
Kavitha Nair,182
Suresh Rao,182
Deepak Sharma,178
Rajesh Kumar,175
Amit Patel,154
Arjun Iyer,152
Not Assigned,75


**Save GOLD table**

In [0]:
technician_performance.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_technician_performance")

In [0]:
display(
spark.table("workspace.default.gold_technician_performance")
)

technician_name,total_jobs
Priya Singh,203
Meena Reddy,199
Suresh Rao,182
Kavitha Nair,182
Deepak Sharma,178
Rajesh Kumar,175
Amit Patel,154
Arjun Iyer,152
Not Assigned,75


Technician Performance Analysis

This report calculates the total number of service jobs handled by each technician. The results help identify the most active technicians and can be used to evaluate workload distribution within the service center.

**Job Status Analysis**
Q2.How many jobs are:

Completed
Pending
Cancelled

In [0]:
from pyspark.sql.functions import count

job_status = silver_df.groupBy("job_status") \
    .agg(
        count("job_id").alias("total_jobs")
    )

display(job_status)

job_status,total_jobs
Completed,1054
Cancelled,76
Pending,111
In Progress,259


In [0]:
%sql

SELECT

job_status,

COUNT(job_id) AS total_jobs

FROM service_data

GROUP BY job_status;

job_status,total_jobs
Completed,1054
Cancelled,76
Pending,111
In Progress,259


In [0]:
job_status.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_job_status")

Job Status Analysis

This report summarizes the number of service jobs according to their current status. It provides a quick overview of completed, pending, and cancelled jobs, helping management monitor the service workflow.

**Delay Analysis**
Q3. how many jobs are completed:
ON TIME 
DElay

In [0]:
delay_analysis = silver_df.groupBy(
    "delivery_status"
).count()

display(delay_analysis)

delivery_status,count
On Time,1203
Delayed,297


In [0]:
%sql

SELECT

delivery_status,

COUNT(*) AS total_jobs

FROM service_data

GROUP BY delivery_status;

delivery_status,total_jobs
On Time,1203
Delayed,297


In [0]:
delay_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_delay_analysis")

Delay Analysis

This report compares the number of jobs completed on time with those delivered after the promised date. It helps evaluate service efficiency and customer satisfaction.

**Repeat Customers**
Q4. Which customers have visited the service center more than once?

In [0]:
from pyspark.sql.functions import count

repeat_customers = silver_df.groupBy(
    "customer_id",
    "customer_name"
).agg(
    count("job_id").alias("total_visits")
).filter("total_visits > 1")

display(repeat_customers)

customer_id,customer_name,total_visits
CUST0295,Meera Khan,5
CUST0108,Rahul Bajaj,9
CUST0001,Kiran Patel,4
CUST0273,Shivam Singh,2
CUST0152,Nisha Gupta,10
CUST0276,Vinay Dubey,8
CUST0034,Imran Verma,6
CUST0142,Yash Pandey,10
CUST0232,Anjali Iyer,5
CUST0181,Deepika Malhotra,6


In [0]:
repeat_customers.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_repeat_customers")

In [0]:
display(
spark.table("workspace.default.gold_repeat_customers")
)

customer_id,customer_name,total_visits
CUST0133,Hema Patel,5
CUST0200,Shreya Khan,5
CUST0066,Manoj Ansari,9
CUST0109,Vikram Malhotra,10
CUST0081,Shreya Khan,8
CUST0102,Tanvi Shah,10
CUST0224,Ajay Khan,6
CUST0196,Kunal Qureshi,3
CUST0137,Anjali Ali,2
CUST0298,Ravi Qureshi,11


This report identifies customers who have visited the service center multiple times. It helps the company understand customer retention and identify customers who frequently use repair services.

**Device Brand Analysis** Q5. Which device brand receives the highest number of repair requests?

In [0]:
from pyspark.sql.functions import count

brand_analysis = silver_df.groupBy(
    "brand"
).agg(
    count("job_id").alias("total_repairs")
).orderBy(
    "total_repairs",
    ascending=False
)

display(brand_analysis)

brand,total_repairs
OnePlus,160
Lenovo,145
Xiaomi,137
LG,136
Acer,134
Motorola,108
Samsung,99
Realme,95
Oppo,88
Vivo,76


In [0]:
brand_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_brand_analysis")

This report shows the number of repair requests received for each device brand. It helps identify brands that require frequent servicing and can support inventory planning.

**Issue Type Analysis**
Q6. Which repair issue occurs most frequently?

Examples:

Battery Issue
Screen Damage
Water Damage
Software Problem

In [0]:
issue_analysis = silver_df.groupBy(
    "issue_type"
).agg(
    count("job_id").alias("total_cases")
).orderBy(
    "total_cases",
    ascending=False
)

display(issue_analysis)

issue_type,total_cases
Charging Port Fault,124
Wi-Fi Not Connecting,121
Software Crash,113
Water Damage,110
Keyboard Fault,106
Hard Disk Failure,105
Screen Damage,101
Overheating,100
Motherboard Failure,95
Battery Issue,94


In [0]:
issue_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_issue_analysis")

This report summarizes the frequency of different repair issues. It helps the service center identify common problems and maintain spare parts accordingly.

**Monthly Repair Trend**
Q7. How many repair jobs were received each month?

In [0]:
from pyspark.sql.functions import month

monthly_repairs = silver_df.withColumn(
    "month",
    month("received_date")
).groupBy(
    "month"
).agg(
    count("job_id").alias("total_repairs")
).orderBy("month")

display(monthly_repairs)

month,total_repairs
1,608
2,528
3,364


In [0]:
monthly_repairs.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_monthly_repairs")

This report displays the number of repair requests received each month. It helps management identify seasonal trends and plan technician availability accordingly.

**Warranty Analysis**
Q8. How many repairs were covered under warranty?

In [0]:
silver_df.printSchema()

root
 |-- device_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- job_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- job_status: string (nullable = true)
 |-- received_date: date (nullable = true)
 |-- promised_date: date (nullable = true)
 |-- completed_date: date (nullable = true)
 |-- technician_id: string (nullable = true)
 |-- technician_name: string (nullable = true)
 |-- repair_notes: string (nullable = true)
 |-- estimated_cost: double (nullable = true)
 |-- actual_cost: double (nullable = true)
 |-- repair_duration: integer (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- brand: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- model_series: string (nullable = true)
 |-- war

In [0]:
warranty_analysis = silver_df.groupBy(
    "warranty_status"
).agg(
    count("job_id").alias("total_jobs")
)

display(warranty_analysis)

warranty_status,total_jobs
Under Warranty,1500


In [0]:
silver_df.select("warranty_months").show(20)

+---------------+
|warranty_months|
+---------------+
|             12|
|             24|
|             12|
|             18|
|              6|
|              6|
|             12|
|             18|
|             18|
|              6|
|             18|
|             12|
|              6|
|             24|
|             24|
|              6|
|             12|
|             24|
|             24|
|             12|
+---------------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import when

silver_df = silver_df.withColumn(
    "warranty_status",
    when(silver_df.warranty_months > 0, "Under Warranty")
    .otherwise("Out of Warranty")
)

In [0]:
from pyspark.sql.functions import count

warranty_analysis = silver_df.groupBy(
    "warranty_status"
).agg(
    count("job_id").alias("total_jobs")
)

display(warranty_analysis)

warranty_status,total_jobs
Under Warranty,1500


In [0]:
warranty_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_warranty_analysis")

In [0]:
display(
    spark.table("workspace.default.gold_warranty_analysis")
)

warranty_status,total_jobs
Under Warranty,1500


This report clearly shows that 1500 jobs are under warranty.

**Repair Cost Analysis**
Q9. Which device brands generate the highest repair costs?

In [0]:
from pyspark.sql.functions import sum

repair_cost = silver_df.groupBy("brand").agg(
    sum("estimated_cost").alias("total_estimated_cost"),
    sum("actual_cost").alias("total_actual_cost")
).orderBy("total_actual_cost", ascending=False)

display(repair_cost)

brand,total_estimated_cost,total_actual_cost
OnePlus,624972.9400000003,537342.11
LG,591299.58,438211.8099999998
Acer,552443.6099999999,437495.75000000006
Lenovo,554406.1200000001,406276.7599999999
Xiaomi,532160.38,396964.33000000013
Motorola,435018.53000000026,387452.3599999999
Samsung,404232.2599999999,327605.3500000001
Realme,441892.3699999999,309964.7
Oppo,424386.59000000014,256108.82000000007
Vivo,305565.86,229897.8600000001


In [0]:
repair_cost.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_repair_cost")

**Technician Workload**
Q10. How many jobs are assigned to each technician?

In [0]:
from pyspark.sql.functions import count

technician_workload = silver_df.groupBy(
    "technician_name"
).agg(
    count("job_id").alias("jobs_assigned")
).orderBy(
    "jobs_assigned",
    ascending=False
)

display(technician_workload)

technician_name,jobs_assigned
Priya Singh,203
Meena Reddy,199
Kavitha Nair,182
Suresh Rao,182
Deepak Sharma,178
Rajesh Kumar,175
Amit Patel,154
Arjun Iyer,152
Not Assigned,75


In [0]:
technician_workload.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_technician_workload")

**Average Repair Duration**
Q11. Which device brands take the longest time to repair?

In [0]:
from pyspark.sql.functions import avg

repair_duration = silver_df.groupBy(
    "brand"
).agg(
    avg("repair_duration").alias("average_repair_days")
).orderBy(
    "average_repair_days",
    ascending=False
)

display(repair_duration)

brand,average_repair_days
Dell,4.8431372549019605
Lenovo,4.69811320754717
LG,4.584158415841584
Samsung,4.546666666666667
HP,4.490566037735849
Motorola,4.426829268292683
Acer,4.425742574257426
Apple,4.42
Oppo,4.39344262295082
OnePlus,4.373015873015873


In [0]:
repair_duration.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_repair_duration")

**City-wise Service Requests**
Q12.Which cities generate the highest number of repair requests?

In [0]:
from pyspark.sql.functions import count

city_analysis = silver_df.groupBy(
    "city"
).agg(
    count("job_id").alias("total_jobs")
).orderBy(
    "total_jobs",
    ascending=False
)

display(city_analysis)

city,total_jobs
Bhopal,149
Pune,137
Delhi,125
Kolkata,118
Hyderabad,117
Lucknow,115
Bengaluru,110
Chennai,98
Mumbai,98
Ahmedabad,94


In [0]:
city_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_city_analysis")

**Business Summary**

In [0]:
from pyspark.sql.functions import count, avg, sum

business_summary = silver_df.agg(
    count("job_id").alias("total_jobs"),
    avg("repair_duration").alias("average_repair_days"),
    sum("actual_cost").alias("total_revenue")
)

display(business_summary)

total_jobs,average_repair_days,total_revenue
1500,4.423893805309734,4741720.220000002


In [0]:
business_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_business_summary")

In [0]:
display(
    spark.table("workspace.default.gold_business_summary")
)

total_jobs,average_repair_days,total_revenue
1500,4.423893805309734,4741720.220000002


In [0]:
spark.sql("SHOW TABLES IN workspace.default").display()

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_devices,false
default,bronze_service_jobs,false
default,gold_brand_analysis,false
default,gold_business_summary,false
default,gold_city_analysis,false
default,gold_delay_analysis,false
default,gold_issue_analysis,false
default,gold_job_status,false
default,gold_monthly_repairs,false


In [0]:
display(
spark.table("workspace.default.gold_business_summary")
)

total_jobs,average_repair_days,total_revenue
1500,4.423893805309734,4741720.220000002


In [0]:
spark.table("gold_warranty_analysis").show()

+---------------+----------+
|warranty_status|total_jobs|
+---------------+----------+
| Under Warranty|      1500|
+---------------+----------+

